# 05 · Kaggle judging and report evidence

Create a benchmark-enabled Kaggle notebook at https://www.kaggle.com/benchmarks/tasks/new, then import/paste these cells. Attach saved comparison outputs; attach training outputs to build plots. This notebook never generates model responses.

Upload this notebook to Kaggle. Enable Internet; select **GPU T4 x2** for training/generation. The judge notebook needs benchmark model access, not a GPU. Enter your public GitHub URL below. No GitHub credentials are needed.


## 1. Configuration

Run All defaults to smoke checks only. After inspecting them, set `RUN_FULL=True`. Use a new run name if changing experimental settings; completed runs are skipped.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = "https://github.com/0x0shephard/genai-pa2.git"
GIT_REF = "main"  # After the first run, use its printed commit SHA for all final runs.
REPO_DIR = Path("/kaggle/working/pa2-repo")
OUTPUT_ROOT = Path("/kaggle/working/outputs")
PREPARED_ROOT = Path("/kaggle/working/prepared")
RUN_SMOKE = True
RUN_FULL = False  # Inspect smoke output, then set True and rerun the configuration/full cells.

PAIRS_ROOT = Path("/kaggle/input/YOUR-GENERATION-OUTPUT/outputs/pairs")
TRACKS = ["hh", "ultrafeedback", "persona"]
JUDGE_MODEL = "google/gemini-3.6-flash"  # Select an available Kaggle judge if unavailable.
JUDGE_RESUME_ROOT = None  # saved outputs/judging containing hh/, ultrafeedback/, persona/
BUILD_REPORT = True
TRAINING_RUNS = {  # Attach these output datasets for plots and actual configurations.
    "sft": "/kaggle/input/YOUR-SFT-OUTPUT/outputs/sft",
    "hh": "/kaggle/input/YOUR-HH-OUTPUT/outputs/dpo_hh",
    "ultrafeedback": "/kaggle/input/YOUR-UF-OUTPUT/outputs/dpo_ultrafeedback",
    "persona": "/kaggle/input/YOUR-PERSONA-OUTPUT/outputs/persona",
}


## 2. Clone the chosen GitHub revision and install dependencies


In [ ]:
assert "YOUR_" not in REPO_URL, "Enter your public GitHub repository URL in the configuration cell."
assert REPO_URL.startswith("https://github.com/"), "Use a public GitHub HTTPS URL."
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    assert origin.rstrip("/").removesuffix(".git") == REPO_URL.rstrip("/").removesuffix(".git"), "Existing checkout belongs to another repository."
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip(), "Checkout has local changes; preserve them before changing revisions."
subprocess.check_call(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
os.environ["ALIGNMENT_GIT_REF"] = GIT_REF
os.environ["PYTHONPATH"] = str(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/tmp/pa2-hf-cache"  # Caches are disposable, not saved notebook artifacts.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
print("Git commit:", COMMIT)

def run(*args):
    subprocess.check_call([sys.executable, "-m", "alignment", *map(str, args)], cwd=REPO_DIR)

def need(path, label):
    path = Path(path)
    if not path.exists():
        print("Available attachments:", list(Path("/kaggle/input").glob("*/*")))
        raise FileNotFoundError(f"Attach the saved {label} output and update its configuration path: {path}")
    return path

def completed(path):
    summary = Path(path) / "summary.json"
    return summary.exists() and json.loads(summary.read_text()).get("completed", False)

def show_samples(path, count=5):
    for row in json.loads(Path(path).read_text())[:count]:
        print("PROMPT:", row["instruction"])
        print("RESPONSE:", row["output"], "\n")


In [ ]:
# Keep the benchmark runtime and its model-proxy dependencies intact.
subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml", "matplotlib"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"])
sys.path.insert(0, str(REPO_DIR))
import kaggle_benchmarks as kbench
print("Kaggle model registry:", kbench.llms)


## 3. Preflight and judge smoke check

Requests use the Kaggle SDK in this notebook kernel so benchmark context remains available. The first two pairs test access and JSON parsing. Each attempt waits at least five seconds.


In [ ]:
from alignment.judge import judge_pairs
from alignment.utils import read_jsonl, write_jsonl
for track in TRACKS:
    need(PAIRS_ROOT / f"{track}.jsonl", f"{track} pairwise comparison")
if RUN_SMOKE:
    track = TRACKS[0]
    smoke_pairs = OUTPUT_ROOT / "judge_smoke_pairs.jsonl"
    write_jsonl(smoke_pairs, read_jsonl(PAIRS_ROOT / f"{track}.jsonl")[:2])
    summary = judge_pairs(smoke_pairs, OUTPUT_ROOT / "judge_smoke", judge_model=JUDGE_MODEL)
    print(json.dumps(summary, indent=2))
    assert summary["quality"]["failures"] == 0, "Resolve model access/parsing failures before the full run."


## 4. Full resumable judging


In [ ]:
if RUN_FULL:
    for track in TRACKS:
        resume = Path(JUDGE_RESUME_ROOT) / track if JUDGE_RESUME_ROOT else None
        summary = judge_pairs(PAIRS_ROOT / f"{track}.jsonl", OUTPUT_ROOT / "judging" / track,
            judge_model=JUDGE_MODEL, batch_size=5, sleep_seconds=5, max_retries=3, resume_from=resume)
        print(json.dumps(summary, indent=2))
else:
    print("Smoke only. Set RUN_FULL=True once judge access and parsing work.")


## 5. Build plots, tables, examples, and a report draft

Complete the draft's interpretation and limitations using the actual results. Export the final PDF and check it is at most eight pages including references.


In [ ]:
if RUN_FULL and BUILD_REPORT:
    from alignment.report import build_report
    for name, directory in TRAINING_RUNS.items():
        need(Path(directory) / "run.json", f"{name} training run")
    build_report(TRAINING_RUNS, {t: OUTPUT_ROOT / "judging" / t for t in TRACKS},
        OUTPUT_ROOT / "report", {t: PAIRS_ROOT / f"{t}.jsonl" for t in TRACKS})
    print((OUTPUT_ROOT / "report/report_draft.md").read_text())


## Preserve outputs

Use **Save Version → Save & Run All** after setting `RUN_FULL=True`, or save completed interactive outputs with the appropriate Kaggle version option. Attach that saved notebook's output to the next stage using **Add Input → Notebook Output**. Confirm the files exist in the saved version before ending the session.

New artifacts are under `/kaggle/working/outputs` and `/kaggle/working/prepared`. Model files are large: do not duplicate them into zip archives unless disk permits. For interruption recovery, attach the prior saved output and configure its resume path. A file in a temporary session is preserved only when Kaggle saves the output.
